# Clustered uncertainty, influence and selection stability

**Purpose:** Compute participant-cluster intervals, influence, balanced metrics and selection summaries.

**Required inputs:** 04, 05, 09 and 11 results ZIPs.

**Outputs:** Bootstrap, influence and selection-stability records.

**Usage:** Run after the named upstream results are available. Files are read from the notebook’s current working directory unless specified in its configuration.


## Inputs and matched trial checks


In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import zipfile

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

NOTEBOOK_NAME = '14_Bacek_Clustered_Uncertainty_Influence_Selection_Stability.ipynb'
FILES = {
    '04': Path('04_Bacek_Foundational_Predictive_Models_results.zip'),
    '05': Path('05_Bacek_Incremental_Gait_Information_results.zip'),
    '09': Path('09_Bacek_Ridge_vs_RBF_SVR_Comparison_results.zip'),
    '11': Path('11_Bacek_Known_Person_Calibration_results.zip'),
}
OUTPUT_DIR = Path('14_Bacek_Clustered_Uncertainty_Influence_Selection_Stability_results')
ZIP_OUT = Path(str(OUTPUT_DIR) + '.zip')
EXPECTED_SOURCE_SHA = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
N_BOOT = 10000
SEED = 2026
checks = []

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def check(name, condition):
    ok = bool(condition)
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL'})
    if not ok:
        raise AssertionError(name)

def close(a, b, atol=1e-11):
    return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float),
                            rtol=0, atol=atol, equal_nan=False))

def read_csv(archive, name):
    return pd.read_csv(BytesIO(archive.read(name)), float_precision='round_trip')

def rmse(y, p):
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    return float(np.sqrt(np.mean(np.square(p-y))))

def metric_summary(y, p):
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    e = p - y
    den = float(np.sum((y-np.mean(y)) ** 2))
    return {
        'RMSE_W_per_kg': rmse(y, p),
        'MAE_W_per_kg': float(np.mean(np.abs(e))),
        'bias_W_per_kg': float(np.mean(e)),
        'R2': float(1 - np.sum(e * e)/den) if den > 0 else float('nan'),
    }

def participant_balanced_rmse(frame, prediction_col):
    per_person = [
        np.mean((g[prediction_col].to_numpy(dtype=np.float64) -
                 g.observed_W_per_kg.to_numpy(dtype=np.float64)) ** 2)
        for _, g in frame.groupby('subject_id', sort=True)
    ]
    return float(np.sqrt(np.mean(per_person)))

for key, path in FILES.items():
    check(f'ZIP {key} exists', path.is_file())
sha = {key: sha256_file(path) for key, path in FILES.items()}

archives = {key: zipfile.ZipFile(path) for key, path in FILES.items()}
summaries = {key: json.loads(archives[key].read('run_summary.json')) for key in FILES}
for key in FILES:
    check(f'Notebook {key} evidence reports PASS', summaries[key].get('status') == 'PASS')
    check(f'Notebook {key} source fingerprint', summaries[key].get('source_sha256') == EXPECTED_SOURCE_SHA)
for key in ('05', '09', '11'):
    check(f'Notebook {key} references the exact Notebook 04 archive',
          summaries[key].get('reference_notebook04_zip_sha256') == sha['04'])

print('Source ZIP fingerprints verified.')
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| pandas:', pd.__version__)

## Participant-cluster bootstrap


In [ ]:
# Population results: B0 and G from Notebook 04; C and C+G from Notebook 05.
ledger04 = read_csv(archives['04'], 'eligible_trial_ledger.csv')
p04 = read_csv(archives['04'], 'held_out_predictions.csv')
p05 = read_csv(archives['05'], 'held_out_predictions.csv')
p09 = read_csv(archives['09'], 'held_out_predictions.csv')

check('284 unique primary trial keys', len(ledger04) == 284 and ledger04.trial_uid.is_unique)
check('21 primary participants', ledger04.subject_id.nunique() == 21)
check('60,757 primary retained cycles', int(ledger04.retained_cycles.sum()) == 60757)

all_pop = ledger04[['trial_uid', 'subject_id', 'outer_fold', 'observed_W_per_kg']].copy()
for archive_name, df, procedure, dest in [
    ('04', p04, 'B0', 'pred_B0'),
    ('04', p04, 'G_SELECTED', 'pred_G'),
    ('05', p05, 'C', 'pred_C'),
    ('05', p05, 'C+G', 'pred_C_plus_G'),
]:
    cur = df[df.procedure == procedure].copy()
    check(f'{archive_name} {procedure} has 284 unique predictions',
          len(cur) == 284 and cur.trial_uid.is_unique)
    check(f'{archive_name} {procedure} has finite predictions and targets',
          np.isfinite(cur[['predicted_W_per_kg', 'observed_W_per_kg']].to_numpy(dtype=float)).all())
    # Check the independently saved trial targets and fold identities before joining.
    joined = all_pop[['trial_uid','subject_id','outer_fold','observed_W_per_kg']].merge(
        cur[['trial_uid','subject_id','outer_fold','observed_W_per_kg','predicted_W_per_kg']],
        on='trial_uid', how='outer', indicator=True, suffixes=('_base','_upstream'), validate='one_to_one')
    check(f'{archive_name} {procedure} trial and participant alignment',
          joined['_merge'].eq('both').all() and
          joined.subject_id_base.eq(joined.subject_id_upstream).all() and
          joined.outer_fold_base.eq(joined.outer_fold_upstream).all() and
          close(joined.observed_W_per_kg_base, joined.observed_W_per_kg_upstream, atol=0))
    all_pop = all_pop.merge(cur[['trial_uid','predicted_W_per_kg']].rename(
        columns={'predicted_W_per_kg':dest}), on='trial_uid', how='left', validate='one_to_one')

ref_c = p04[p04.procedure == 'B2'].set_index('trial_uid').sort_index()
ref_g = p05[p05.procedure == 'G'].set_index('trial_uid').sort_index()
check('C reproduces B2 prediction from Notebook 04', close(
    all_pop.set_index('trial_uid').sort_index().pred_C,
    ref_c.predicted_W_per_kg, atol=0))
check('G reproduces Notebook 05 reused reference', close(
    all_pop.set_index('trial_uid').sort_index().pred_G,
    ref_g.predicted_W_per_kg, atol=0))

# Personal calibration cohort: identical 58 evaluation trials under both K budgets.
p11 = read_csv(archives['11'], 'held_out_predictions.csv')
personal = None
person_procs = [
    'A_K3', 'B0_PERSON_K3', 'B1_PERSON_K3', 'C_CAL_K3',
    'A_K6', 'B0_PERSON_K6', 'B1_PERSON_K6', 'C_CAL_K6',
]
for proc in person_procs:
    x = p11[p11.procedure == proc].copy()
    check(f'{proc}: 58 unique personal evaluation predictions',
          len(x) == 58 and x.trial_uid.is_unique and x.subject_id.nunique() == 20)
    if personal is None:
        personal = x[['trial_uid','subject_id','outer_fold','observed_W_per_kg']].copy()
    joined = personal.merge(x, on='trial_uid', how='outer', indicator=True,
                            suffixes=('_base','_new'), validate='one_to_one')
    check(f'{proc}: same held-out personal trials and observed values',
          joined['_merge'].eq('both').all() and
          joined.subject_id_base.eq(joined.subject_id_new).all() and
          joined.outer_fold_base.eq(joined.outer_fold_new).all() and
          close(joined.observed_W_per_kg_base, joined.observed_W_per_kg_new, atol=0))
    personal = personal.merge(x[['trial_uid','predicted_W_per_kg']].rename(
        columns={'predicted_W_per_kg':'pred_' + proc}),
        on='trial_uid', how='left', validate='one_to_one')

check('20-person personal cohort, with S12 excluded',
      personal.subject_id.nunique() == 20 and 'S12' not in set(personal.subject_id))
check('Population A is identical at K=3 and K=6',
      close(personal.pred_A_K3, personal.pred_A_K6, atol=0))
# Population A must exactly reuse Notebook 04 D-ALL9 predictions for these trials.
ref_a = p04[p04.procedure == 'D-ALL9_FIXED'].set_index('trial_uid')
check('Population A reuses fixed-D Notebook 04 predictions', close(
      personal.pred_A_K3, ref_a.loc[personal.trial_uid, 'predicted_W_per_kg'], atol=1e-12))

for data, name in [(all_pop,'population'), (personal,'personal')]:
    check(name + ': all saved numerical predictions finite',
          np.isfinite(data.select_dtypes(include='number').to_numpy(dtype=float)).all())

print('Population:', len(all_pop), 'trials /', all_pop.subject_id.nunique(), 'participants')
print('Personal:', len(personal), 'trials /', personal.subject_id.nunique(), 'participants')

## Participant-level evaluation


In [ ]:
metric_rows = []
for cohort, data, procedures in [
    ('population', all_pop, {'B0':'pred_B0','G':'pred_G','C':'pred_C','C+G':'pred_C_plus_G'}),
    ('personal_K3', personal, {p:p and 'pred_'+p for p in person_procs[:4]}),
    ('personal_K6', personal, {p:p and 'pred_'+p for p in person_procs[4:]}),
]:
    for procedure, col in procedures.items():
        y, p = data.observed_W_per_kg.to_numpy(), data[col].to_numpy()
        metric_rows.append({'cohort':cohort, 'procedure':procedure, 'n_trials':len(data),
            'n_participants':data.subject_id.nunique(),
            **metric_summary(y, p),
            'participant_balanced_RMSE_W_per_kg':participant_balanced_rmse(data, col)})
metrics = pd.DataFrame(metric_rows)

saved04 = read_csv(archives['04'], 'pooled_metrics.csv').set_index('procedure')
saved05 = read_csv(archives['05'], 'pooled_metrics.csv').set_index('procedure')
saved11 = read_csv(archives['11'], 'pooled_metrics.csv').set_index('procedure')
for proc, source, stored_proc in [
    ('B0',saved04,'B0'), ('G',saved04,'G_SELECTED'),
    ('C',saved05,'C'), ('C+G',saved05,'C+G')]:
    out = metrics[(metrics.cohort=='population') & (metrics.procedure==proc)].iloc[0]
    check(proc + ': pooled RMSE agrees with upstream evidence',
          close(out.RMSE_W_per_kg, source.loc[stored_proc,'RMSE_W_per_kg']))
for proc in person_procs:
    out = metrics[metrics.procedure==proc].iloc[0]
    check(proc + ': pooled RMSE agrees with E1.3 evidence',
          close(out.RMSE_W_per_kg, saved11.loc[proc,'rmse_W_per_kg']))

print(metrics[['cohort','procedure','RMSE_W_per_kg',
               'participant_balanced_RMSE_W_per_kg']].to_string(index=False, float_format=lambda x: f'{x:.6f}'))

## Influence and selection


In [ ]:
POP_PROCS = {'B0':'pred_B0', 'G':'pred_G', 'C':'pred_C', 'C+G':'pred_C_plus_G'}
PERSON_PROCS = {p:'pred_'+p for p in ['A_K3', 'C_CAL_K3','A_K6','C_CAL_K6']}

def cluster_sums(data, proc_columns):
    ids = sorted(data.subject_id.unique())
    counts = np.asarray([(data.subject_id == sid).sum() for sid in ids], dtype=np.int64)
    sq = {}
    for name, col in proc_columns.items():
        diff = data[col].to_numpy(dtype=np.float64) - data.observed_W_per_kg.to_numpy(dtype=np.float64)
        sq[name] = np.asarray([np.sum(diff[data.subject_id.to_numpy() == sid] ** 2,
                                              dtype=np.float64) for sid in ids])
    return ids, counts, sq

pop_ids, pop_count, pop_sq = cluster_sums(all_pop, POP_PROCS)
pers_ids, pers_count, pers_sq = cluster_sums(personal, PERSON_PROCS)
check('Population bootstrap has 21 cluster IDs', len(pop_ids)==21)
check('Personal bootstrap has 20 cluster IDs', len(pers_ids)==20)
check('Population cluster counts sum to 284', pop_count.sum()==284)
check('Personal cluster counts sum to 58', pers_count.sum()==58)

rng = np.random.default_rng(SEED)
bit_generator = type(rng.bit_generator).__name__
pop_draw_indices = rng.integers(0,len(pop_ids),size=(N_BOOT,len(pop_ids)))
pers_draw_indices = rng.integers(0,len(pers_ids),size=(N_BOOT,len(pers_ids)))
check('Exactly 10,000 population cluster draws', pop_draw_indices.shape==(10000,21))
check('Exactly 10,000 personal cluster draws', pers_draw_indices.shape==(10000,20))

def boot_rmse(per_subject_sse, n_each, draw_indices):
    return np.sqrt(np.sum(per_subject_sse[draw_indices],axis=1,dtype=np.float64) /
                   np.sum(n_each[draw_indices],axis=1,dtype=np.float64))

bootvals = {'draw_index':np.arange(1, N_BOOT+1,dtype=int)}
for name, arr in pop_sq.items():
    bootvals['RMSE_'+name] = boot_rmse(arr, pop_count, pop_draw_indices)
for name, arr in pers_sq.items():
    bootvals['RMSE_'+name] = boot_rmse(arr, pers_count, pers_draw_indices)

contrasts = [
    ('B0_minus_G','primary_population','population','B0','G'),
    ('C_minus_CplusG','primary_population','population','C','C+G'),
    ('A_minus_Ccal_K3','secondary_personal','personal','A_K3','C_CAL_K3'),
    ('A_minus_Ccal_K6','secondary_personal','personal','A_K6','C_CAL_K6'),
]
interval_rows = []
for ident, priority, cohort, reference, candidate in contrasts:
    data = all_pop if cohort=='population' else personal
    cols = POP_PROCS if cohort=='population' else PERSON_PROCS
    point = rmse(data.observed_W_per_kg, data[cols[reference]]) - rmse(
        data.observed_W_per_kg, data[cols[candidate]])
    values = bootvals['RMSE_'+reference] - bootvals['RMSE_'+candidate]
    bootvals['delta_'+ident] = values
    lo, hi = np.percentile(values, [2.5,97.5])
    interval_rows.append(dict(comparison=ident, status=priority, reference=reference,
        candidate=candidate, cohort=cohort, n_participants=data.subject_id.nunique(),
        n_trials=len(data), point_delta_RMSE_W_per_kg=float(point),
        lower_95pct_W_per_kg=float(lo), upper_95pct_W_per_kg=float(hi),
        bootstrap_resamples=N_BOOT, seed=SEED))
intervals = pd.DataFrame(interval_rows)
bootstrap_statistics = pd.DataFrame(bootvals)
check('All bootstrap statistics finite', np.isfinite(
      bootstrap_statistics.select_dtypes(include='number').to_numpy()).all())
check('All four 95% intervals ordered',
      (intervals.lower_95pct_W_per_kg <= intervals.upper_95pct_W_per_kg).all())

# Check bootstrap arithmetic on randomly generated cluster-index rows against
# explicit expansion by sampled participant IDs (including duplicate participants).
for indices, data, proc, ids, group_col in [
    (pop_draw_indices, all_pop, 'pred_G', pop_ids, 'RMSE_G'),
    (pers_draw_indices, personal, 'pred_C_CAL_K3', pers_ids, 'RMSE_C_CAL_K3'),
]:
    for i in (0, 7, 500, 9999):
        drawn_ids = [ids[j] for j in indices[i]]
        n = 0
        ss = 0.0
        for sid in drawn_ids:
            subset = data[data.subject_id == sid]
            residual = subset[proc].to_numpy()-subset.observed_W_per_kg.to_numpy()
            ss += float(np.dot(residual,residual))
            n += len(subset)
        check(f'{group_col}: explicit repeated-cluster draw {i+1}',
              close(np.sqrt(ss/n), bootstrap_statistics.iloc[i][group_col], atol=1e-12))

bootstrap_draws = pd.DataFrame({
    'draw_index': np.arange(1,N_BOOT+1,dtype=int),
    'population_sampled_participants': [';'.join(pop_ids[j] for j in row) for row in pop_draw_indices],
    'population_distinct_participants': [len(set(row)) for row in pop_draw_indices],
    'personal_sampled_participants': [';'.join(pers_ids[j] for j in row) for row in pers_draw_indices],
    'personal_distinct_participants': [len(set(row)) for row in pers_draw_indices],
})
check('All resample IDs archived',len(bootstrap_draws)==N_BOOT and
      bootstrap_draws.population_sampled_participants.str.count(';').eq(20).all() and
      bootstrap_draws.personal_sampled_participants.str.count(';').eq(19).all())
print(intervals[['comparison','point_delta_RMSE_W_per_kg',
                 'lower_95pct_W_per_kg','upper_95pct_W_per_kg']].to_string(index=False,float_format=lambda x:f'{x:+.5f}'))

## Paired-comparison intervals


In [ ]:
def primary_metrics(data):
    y = data.observed_W_per_kg
    b0 = rmse(y,data.pred_B0)
    g = rmse(y,data.pred_G)
    c = rmse(y,data.pred_C)
    cg = rmse(y,data.pred_C_plus_G)
    return {'G_RMSE':g, 'B0_minus_G':b0-g, 'C_minus_CplusG':c-cg}

full_values = primary_metrics(all_pop)
influence_rows=[]
for sid in pop_ids:
    omitted = all_pop[all_pop.subject_id != sid]
    without = primary_metrics(omitted)
    for name, full in full_values.items():
        influence_rows.append({'subject_id':sid,'metric':name,
            'full_value_W_per_kg':full,'without_subject_W_per_kg':without[name],
            'influence_full_minus_omitted_W_per_kg':full-without[name],
            'n_omitted_trials':int((all_pop.subject_id==sid).sum())})
influence = pd.DataFrame(influence_rows)
check('63 population influence estimates (21 people × 3 metrics)',len(influence)==63)
check('G influence matches full-population RMSE',close(
      full_values['G_RMSE'], metrics[(metrics.cohort=='population') &
                                  (metrics.procedure=='G')].iloc[0].RMSE_W_per_kg))

# Selected settings in the primary gait, condition+gait and two fixed SVR comparisons.
s04 = read_csv(archives['04'],'selected_models.csv')
s05 = read_csv(archives['05'],'selected_models.csv')
s09 = read_csv(archives['09'],'selected_models.csv')
selection_frames = []
for data, label, proc in [(s04,'G','G_SELECTED'),(s05,'C+G','C+G'),
                           (s09,'W-ALL9 SVR','W-ALL9_SVR'),(s09,'D-ALL9 SVR','D-ALL9_SVR')]:
    df = data[data.procedure==proc].copy()
    check(label+': exactly five selected outer-fold models',
          len(df)==5 and set(df.outer_fold)=={1,2,3,4,5})
    for field in ('representation','alpha','n_components','C','epsilon'):
        if field not in df: df[field] = np.nan
    df['analysis_procedure'] = label
    selection_frames.append(df[['analysis_procedure','outer_fold','representation',
                                'alpha','n_components','C','epsilon']])
selection = pd.concat(selection_frames,ignore_index=True).sort_values(
    ['analysis_procedure','outer_fold']).reset_index(drop=True)
selection_counts = (selection.groupby(['analysis_procedure','representation'],dropna=False)
                    .size().rename('n_outer_folds').reset_index())
check('All 20 cross-fold selection records retained',len(selection)==20)

# Held-out known-person calibration improvement for each individual, on identical trials.
heterogeneity=[]
for k in (3,6):
    for sid, part in personal.groupby('subject_id',sort=True):
        a=rmse(part.observed_W_per_kg,part['pred_A_K'+str(k)])
        c=rmse(part.observed_W_per_kg,part['pred_C_CAL_K'+str(k)])
        heterogeneity.append({'subject_id':sid,'K':k,'n_evaluation_trials':len(part),
            'RMSE_A_W_per_kg':a,'RMSE_C_cal_W_per_kg':c,
            'delta_A_minus_Ccal_W_per_kg':a-c,
            'calibration_improved_RMSE':bool(a>c)})
heterogeneity=pd.DataFrame(heterogeneity)
check('40 participant-by-budget calibration summaries',len(heterogeneity)==40)

print('Population influence (largest absolute changes):')
print(influence.assign(abs_change=lambda d:abs(d.influence_full_minus_omitted_W_per_kg))
      .sort_values('abs_change',ascending=False).head(6).drop(columns='abs_change').to_string(index=False))
print('\nG-selected representation frequencies:')
print(selection_counts[selection_counts.analysis_procedure=='G'].to_string(index=False))
print('\nPersonal calibration improvements:')
print(heterogeneity.groupby('K')['calibration_improved_RMSE'].agg(['sum','count']).to_string())

## Export saved evidence


In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR=OUTPUT_DIR/'figures'
FIG_DIR.mkdir(exist_ok=True)
figure_intervals = intervals[intervals.status=='primary_population'].copy()
fig, ax = plt.subplots(figsize=(7,2.8))
locations = [1,0]
for ypos, (_, row) in zip(locations, figure_intervals.iterrows()):
    midpoint=row.point_delta_RMSE_W_per_kg
    ax.errorbar(midpoint,ypos,xerr=[[midpoint-row.lower_95pct_W_per_kg],
                                    [row.upper_95pct_W_per_kg-midpoint]],
                fmt='o',capsize=4)
ax.axvline(0,linestyle='--',linewidth=1)
ax.set_yticks(locations, ['B0 − G','C − C+G'])
ax.set_xlabel('Paired RMSE improvement (W/kg); positive favours second procedure')
ax.set_title('Participant-cluster bootstrap: primary comparisons (95% intervals)')
ax.set_ylim(-0.6,1.6)
ax.grid(axis='x',alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR/'E5_2_Primary_Paired_Bootstrap_Intervals.pdf',bbox_inches='tight')
fig.savefig(FIG_DIR/'E5_2_Primary_Paired_Bootstrap_Intervals.png',dpi=220,bbox_inches='tight')
plt.show()

## Save traceable evidence and the ZIP archive


In [ ]:
exports = {
    'population_matched_predictions.csv':all_pop,
    'personal_matched_predictions.csv':personal,
    'participant_balanced_metrics.csv':metrics,
    'paired_bootstrap_intervals.csv':intervals,
    'bootstrap_resampled_participants.csv':bootstrap_draws,
    'bootstrap_statistics.csv':bootstrap_statistics,
    'participant_influence.csv':influence,
    'selection_stability.csv':selection,
    'selection_frequencies.csv':selection_counts,
    'personal_calibration_heterogeneity.csv':heterogeneity,
}
for name, df in exports.items():
    df.to_csv(OUTPUT_DIR/name,index=False,float_format='%.17g')

# Provenance: hash source code cells in the notebook on disk, if available.
nb_path=Path(NOTEBOOK_NAME)
if nb_path.is_file():
    nb_source=json.loads(nb_path.read_text(encoding='utf-8'))
    code_sources=[''.join(cell.get('source',[])) if isinstance(cell.get('source',[]),list)
                  else cell.get('source','') for cell in nb_source['cells']
                  if cell.get('cell_type')=='code']
    code_fingerprint=hashlib.sha256('\n\n'.join(code_sources).encode('utf-8')).hexdigest()
else:
    code_fingerprint=None

run_summary={
    'status':'PASS', 'experiment':'E5.2',
    'source_sha256':EXPECTED_SOURCE_SHA,
    'evaluation_setting':'internally held-out, exploratory-informed; fixed predictions',
    'target':'stored corrected trial-level steady-state net metabolic power, W/kg',
    'primary_population_trials':len(all_pop),'primary_population_participants':len(pop_ids),
    'personal_evaluation_trials':len(personal),'personal_participants':len(pers_ids),
    'bootstrap_resamples':N_BOOT,'seed':SEED,
    'random_generator':'numpy.random.default_rng','numpy_bit_generator':bit_generator,
    'bootstrap_method':'participant-cluster resampling with replacement, whole matching trial clusters and multiplicity',
    'confidence_interval':'95% percentile interval, 2.5th and 97.5th percentiles',
    'paired_draws':'same sampled person IDs for reference and candidate in each contrast',
    'predictions_fixed':True,'model_refitting_included':False,
    'source_archives_sha256':{FILES[k].name:sha[k] for k in FILES},
    'upstream_notebook_code_sha256':{k:summaries[k].get('notebook_code_sha256') for k in FILES},
    'notebook_filename':NOTEBOOK_NAME,'notebook_code_sha256':code_fingerprint,
    'software':{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,
                'matplotlib':matplotlib.__version__,'os':platform.platform()},
    'point_estimates_and_intervals':intervals.to_dict('records'),
    'verification':{'n_checks':len(checks),'n_passed':sum(x['status']=='PASS' for x in checks)},
    'completed_at_utc':datetime.now(timezone.utc).isoformat(),
}
with open(OUTPUT_DIR/'run_summary.json','w',encoding='utf-8') as f:
    json.dump(run_summary,f,indent=2,ensure_ascii=False)
pd.DataFrame(checks).to_csv(OUTPUT_DIR/'verification_checks.csv',index=False)
with zipfile.ZipFile(ZIP_OUT,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for file in sorted(OUTPUT_DIR.rglob('*')):
        if file.is_file():
            z.write(file,arcname=file.relative_to(OUTPUT_DIR).as_posix())
for archive in archives.values():
    archive.close()
print('\nSaved',ZIP_OUT,'(',ZIP_OUT.stat().st_size,'bytes )')
print('Verification:',len(checks),'/',len(checks),'PASS')
print('Files in ZIP:',len([q for q in OUTPUT_DIR.rglob('*') if q.is_file()]))